# Milestone 4 — Week 1 Baseline: Loss-Based Membership Inference Attack on VLM

In [ ]:
!pip install -q transformers datasets accelerate Pillow scikit-learn tqdm num2words

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 9.5 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoProcessor, SmolVLMForConditionalGeneration
from PIL import Image
import io
import pandas as pd
from tqdm.notebook import tqdm

print(f"PyTorch     : {torch.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device      : {device}")

## 1. Load Model and Processor

In [ ]:
model_id = "UBC-SLIME/colx_585_vlm"

processor = AutoProcessor.from_pretrained(model_id)
processor.image_processor.do_image_splitting = False
processor.image_processor.size = {"longest_edge": 512}
processor.image_processor.max_image_size = {"longest_edge": 512}

model = SmolVLMForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    _attn_implementation="sdpa",
    trust_remote_code=True
).to(device)
model.eval()
print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/513M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

Model loaded.


## 2. Compute Per-Sample Loss

In [4]:
def load_image(example):
    if isinstance(example['image'], dict) and 'bytes' in example['image']:
        return Image.open(io.BytesIO(example['image']['bytes'])).convert("RGB")
    return example['image'].convert("RGB")


def compute_loss(example, model, processor, device):
    """Return the model's cross-entropy loss for a single image-caption pair."""
    image  = load_image(example)
    parts  = example['text'].split('\n', 1)
    messages = [
        {"role": "user",      "content": [{"type": "image"}, {"type": "text", "text": parts[0]}]},
        {"role": "assistant", "content": [{"type": "text", "text": parts[1] if len(parts) > 1 else ""}]}
    ]
    text   = processor.apply_chat_template(messages, tokenize=False)
    inputs = processor(text=text, images=[image], return_tensors="pt").to(device)
    if device == "cuda":
        inputs = {k: v.to(torch.bfloat16) if v.dtype == torch.float32 else v
                  for k, v in inputs.items()}
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    with torch.no_grad():
        outputs = model(**inputs, labels=labels)
    return outputs.loss.item()

## 3. Score All Splits

In [5]:
dataset_id = "UBC-SLIME/colx585_group_project_data"

def score_split(split_name):
    ds      = load_dataset(dataset_id, split=split_name, streaming=True)
    records = []
    for example in tqdm(ds, desc=f"Scoring {split_name}"):
        try:
            loss   = compute_loss(example, model, processor, device)
            record = {"id": example["id"], "loss": loss, "score": -loss}
            if "is_member" in example:
                record["label"] = example["is_member"]
            if "type" in example:
                record["type"] = example["type"]
            records.append(record)
        except Exception as e:
            print(f"Skipped {example['id']}: {e}")
    return pd.DataFrame(records)

df_train = score_split("train")
df_val   = score_split("validation")
df_test  = score_split("test")

Scoring train: 0it [00:00, ?it/s]

Scoring validation: 0it [00:00, ?it/s]

Scoring test: 0it [00:00, ?it/s]

## 4. Evaluate on Train & Validation Splits

In [6]:
from sklearn.metrics import roc_auc_score, roc_curve

def evaluate(df, split_name):
    if "label" not in df.columns:
        print(f"{split_name}: no labels available (test split).")
        return
    auc = roc_auc_score(df["label"], df["score"])
    fpr, tpr, _ = roc_curve(df["label"], df["score"])
    # TPR at the point where FPR is closest to 0.1
    idx = np.searchsorted(fpr, 0.1)
    tpr_at_fpr01 = tpr[idx]
    print(f"[{split_name:>12s}]  AUC = {auc:.4f}  |  TPR@FPR=0.1 = {tpr_at_fpr01:.4f}")

evaluate(df_train, "train")
evaluate(df_val,   "validation")

[       train]  AUC = 0.5028  |  TPR@FPR=0.1 = 0.1013
[  validation]  AUC = 0.5071  |  TPR@FPR=0.1 = 0.0867


## 5. Kaggle Submission 

In [ ]:
submission = df_test[["id", "score"]].copy()
submission = submission.rename(columns={"score": "score"})

submission.to_csv("submission_baseline.csv", index=False)
print(f"Submission saved: {len(submission)} rows")
submission.head()

Submission saved: 6000 rows


,id,score
0,cauldron_ln_8044,-0.529115
1,cauldron_ln_7863,-0.432044
2,mix_SI_UT_5400_19,-0.768765
3,mix_SI_UT_5400_896,-0.546389
4,cauldron_ln_17842,-0.399349
